# Statistical Inference -- DS Interview Reference (R)

Companion to the *Sampling Methods* notebook. Where that notebook covers **how to
collect data**, this one covers **how to draw reliable conclusions from it**.
Everything is implemented in **base R only** -- `pnorm`, `pt`, `qt`, `sample`,
`rnorm`, `rbinom` -- the way you would in a live coding round.

## Quick Index

| Goal | Section |
| :--- | :--- |
| Point estimates, bias, why n-1, bias-variance tradeoff | SS1 |
| Confidence intervals -- z, t, bootstrap in one place | SS2 |
| Hypothesis testing framework, Type I/II errors | SS3 |
| The peeking problem (why you can't check A/B tests early) | SS4 |
| One-sample z and t tests from scratch | SS5 |
| Two-sample tests + effect size (Welch, two-proportion z) | SS6 |
| Multiple testing correction (Bonferroni, BH/FDR) | SS7 |
| Statistical power via simulation | SS8 |
| Sample size & minimum detectable effect | SS9 |
| End-to-end A/B test design | SS10 |

---

> **Interview note (R vs Python):** The Python source notebook hand-rolls a normal
> CDF (via `math.erfc`) and a t-distribution CDF/inverse-CDF (via a regularized
> incomplete-beta continued fraction) because it deliberately avoids `scipy`. In R,
> none of that machinery is needed -- `pnorm()`, `pt()`, and `qt()` are part of
> **base R**, no package required. This is a case where the R port is genuinely
> shorter than the Python original: the "from scratch" exercise was specifically
> a scipy-avoidance exercise, and base R was never missing the functions in the
> first place.

---
## Shared helpers (used throughout)

In [ ]:
# ── p-value helpers, base R only -- no packages needed ───────────────────────
norm_two_sided_p <- function(z) {
  2 * pnorm(abs(z), lower.tail = FALSE)   # two-sided p-value for a z-statistic
}

t_two_sided_p <- function(t, df) {
  2 * pt(abs(t), df = df, lower.tail = FALSE)   # two-sided p-value for a t-statistic
}

# Critical t-value (the R port's equivalent of the hand-rolled `t_ppf` in Python)
# is just qt() -- base R's inverse t-CDF, no bisection search required.
t_crit <- function(p, df) qt(p, df = df)

# Quick self-check
cat("norm_two_sided_p(1.96) =", round(norm_two_sided_p(1.96), 4), "(expect ~0.05)\n")
cat("qt(0.975, df=39)       =", round(qt(0.975, 39), 4), "(expect ~2.0227)\n")

---
# Part 1 -- Estimation

---
## SS1 -- Point Estimation, Bias, and the Bias-Variance Tradeoff

**Intuition:** An estimator is a recipe for guessing a population quantity from a
sample. Two things matter: is it **biased** (systematically off), and how much does
it **vary** from sample to sample? The deepest idea in this section is that these
trade off against each other -- a slightly biased estimator with low variance can
beat an unbiased one with high variance, measured by mean squared error:

$$\text{MSE} = \text{Bias}^2 + \text{Variance}$$

This is *the* concept behind regularization, shrinkage, and why "unbiased" is not
always the goal.

---

> **Interview prompts:**
> *(a) Why does sample variance divide by n-1 instead of n?*
> *(b) Give an example where a biased estimator is better than an unbiased one.*

In [ ]:
set.seed(42)

# ── (a) Why n-1? Demonstrate the bias empirically ────────────────────────────
population <- rnorm(100000, mean = 50, sd = 10)
true_var   <- mean((population - mean(population))^2)   # population variance, divide-by-n definition

biased <- numeric(5000); unbiased <- numeric(5000)
for (i in 1:5000) {
  s <- sample(population, size = 30, replace = FALSE)
  m <- mean(s)
  biased[i]   <- mean((s - m)^2)        # divide by n   (R has no built-in ddof=0 var())
  unbiased[i] <- var(s)                 # divide by n-1 (R's var() default IS ddof=1, unlike NumPy's default)
}

cat("Why n-1 (Bessel's correction):\n")
cat(sprintf("  True population variance        : %.2f\n", true_var))
cat(sprintf("  E[sample var] with /n  (biased) : %.2f  <- biased LOW\n", mean(biased)))
cat(sprintf("  E[sample var] with /n-1(var())  : %.2f  <- unbiased\n", mean(unbiased)))
cat("  Dividing by n systematically underestimates because the sample mean\n")
cat("  is closer to the sample than the true mean is.\n")

# ── (b) Bias-variance tradeoff: a shrinkage estimator ────────────────────────
true_mean <- 50.0
prior     <- 45.0     # prior guess (slightly wrong)
lam       <- 0.3      # shrinkage strength
n         <- 10

sample_ests <- replicate(5000, mean(rnorm(n, true_mean, 10)))
shrunk_ests <- (1 - lam) * sample_ests + lam * prior

cat("\nBias-variance tradeoff (estimating a mean, n=10):\n")
cat(sprintf("  %-18s %8s %10s %8s\n", "Estimator", "Bias", "Variance", "MSE"))
cat(sprintf("  %-18s %8.3f %10.3f %8.3f\n", "Sample mean",
            mean(sample_ests) - true_mean, var(sample_ests), mean((sample_ests - true_mean)^2)))
cat(sprintf("  %-18s %8.3f %10.3f %8.3f\n", "Shrunk (lam=0.3)",
            mean(shrunk_ests) - true_mean, var(shrunk_ests), mean((shrunk_ests - true_mean)^2)))
cat("\n  The shrinkage estimator is BIASED but has lower VARIANCE and lower MSE.\n")
cat("  This is why regularization (which biases estimates toward zero) often wins.\n")

**Key takeaways:**
- **Bias** = expected estimate minus true value. **Variance** = how much the estimate jumps between samples.
- R's built-in `var()` already divides by n-1 (matches pandas' default `ddof=1`) -- unlike NumPy, whose `.var()` defaults to `ddof=0`. **This is the single biggest gotcha when porting variance code from NumPy to R: the defaults disagree.**
- **MSE = Bias squared + Variance.** Minimizing MSE sometimes means accepting bias to cut variance.
- Shrinkage and regularization are deliberate bias-for-variance trades. This is why "unbiased" is not always the goal.

**Common mistakes:**
- Assuming `var()` in R behaves like `np.var()` in Python -- it doesn't; R's `var()` is always the n-1 (unbiased, sample) version, with no `ddof` argument to change it. If you genuinely need the n-divisor version, compute it manually as shown above
- Equating "unbiased" with "best" -- MSE is usually what you actually care about

---
## SS2 -- Confidence Intervals, Unified

**Intuition:** A 95% confidence interval is a recipe that, across many repeated
samples, captures the true parameter 95% of the time. It is **not** "a 95% chance
the true value is in this particular interval" -- the true value is fixed; the
interval is what's random. There are three workhorse constructions:

| Method | Use when | Formula |
| :--- | :--- | :--- |
| **z-interval** | n large (>= 30), or known variance | $\bar{x} \pm z \cdot s/\sqrt{n}$ |
| **t-interval** | n small, normal-ish data | $\bar{x} \pm t_{\alpha/2,n-1} \cdot s/\sqrt{n}$ |
| **Bootstrap** | any statistic, no assumptions | percentiles of resampled stats |

---

> **Interview prompt:** *"Construct a 95% CI for the mean three ways -- normal
> approximation, t-distribution, and bootstrap -- and explain when each is appropriate."*

In [ ]:
set.seed(42)
data <- rnorm(40, mean = 50, sd = 10)    # small-ish sample
n    <- length(data)
xbar <- mean(data)
s    <- sd(data)                          # R's sd() is already the n-1 version
se   <- s / sqrt(n)

# ── 1. z-interval (normal approximation) ─────────────────────────────────────
z_ci <- c(xbar - 1.96 * se, xbar + 1.96 * se)

# ── 2. t-interval (small-sample correct) ─────────────────────────────────────
tc   <- qt(0.975, df = n - 1)             # base R, no hand-rolled inverse-CDF needed
t_ci <- c(xbar - tc * se, xbar + tc * se)

# ── 3. Bootstrap percentile interval ─────────────────────────────────────────
boot <- replicate(5000, mean(sample(data, n, replace = TRUE)))
b_ci <- quantile(boot, c(0.025, 0.975))

cat(sprintf("Sample mean: %.3f,  SE: %.3f,  n: %d\n", xbar, se, n))
cat(sprintf("\n%-14s %22s %8s\n", "Method", "95% CI", "Width"))
cat(sprintf("%-14s (%.3f, %.3f) %8.3f\n", "z-interval", z_ci[1], z_ci[2], z_ci[2] - z_ci[1]))
cat(sprintf("%-14s (%.3f, %.3f) %8.3f\n", "t-interval", t_ci[1], t_ci[2], t_ci[2] - t_ci[1]))
cat(sprintf("%-14s (%.3f, %.3f) %8.3f\n", "bootstrap", b_ci[1], b_ci[2], b_ci[2] - b_ci[1]))
cat(sprintf("\nt_crit = %.4f vs z = 1.96 -- the t-interval is wider, correctly\n", tc))
cat("reflecting the extra uncertainty from estimating the variance at small n.\n")

# ── Coverage check: does the t-interval really cover 95%? ─────────────────────
covered <- 0
for (i in 1:5000) {
  s2  <- rnorm(n, 50, 10)
  se2 <- sd(s2) / sqrt(n)
  lo  <- mean(s2) - tc * se2
  hi  <- mean(s2) + tc * se2
  if (lo <= 50 && 50 <= hi) covered <- covered + 1
}
cat(sprintf("\nt-interval empirical coverage: %.1f%% (target 95%%)\n", covered / 5000 * 100))

**Common mistakes:**
- Misinterpreting a CI as "95% probability the parameter is in *this* interval" -- the correct frequentist reading is about the long-run behavior of the procedure
- Using the z-interval at small n -- it is too narrow because it ignores the uncertainty in estimating the standard deviation; use t
- Forgetting that the bootstrap CI is the only one that works for non-mean statistics (median, ratios, AUC) without new formulas
- Reaching for `t.test(data, mu = 50)$conf.int` in real code instead of hand-rolling -- base R's built-in already does exactly this; the manual version above is for understanding, not production use

---
# Part 2 -- Hypothesis Testing

---
## SS3 -- The Testing Framework and the Two Error Types

**Intuition:** A hypothesis test asks whether the data is surprising under a "null"
assumption ($H_0$, usually "no effect"). You compute a test statistic, then a
**p-value** -- the probability of seeing data at least this extreme *if $H_0$ were
true*. Small p-value -> the data is surprising under $H_0$ -> reject it.

Two ways to be wrong:

| | $H_0$ true | $H_0$ false |
| :--- | :--- | :--- |
| **Reject $H_0$** | Type I error (false positive), rate $\alpha$ | Correct (power) |
| **Fail to reject** | Correct | Type II error (false negative), rate $\beta$ |

**Power = 1 - beta** is the probability of detecting a real effect.

> **What a p-value is NOT:** it is not the probability that $H_0$ is true, and not
> the probability your result happened by chance. It is P(data this extreme | $H_0$).

---

> **Interview prompt:** *"Simulate to show that your test's Type I error rate
> actually matches alpha, and measure its power against a specific alternative."*

In [ ]:
set.seed(42)
alpha  <- 0.05
n      <- 50
TRIALS <- 5000

# ── Type I error: H0 is TRUE (true mean = 0). How often do we wrongly reject? ─
type1 <- 0
for (i in 1:TRIALS) {
  s <- rnorm(n, 0.0, 1)                  # H0 true: mean = 0
  z <- mean(s) / (sd(s) / sqrt(n))
  if (abs(z) > 1.96) type1 <- type1 + 1  # reject at alpha=0.05
}
cat(sprintf("Type I error rate : %.4f   (target alpha = %s)\n", type1 / TRIALS, alpha))

# ── Type II error & power: H1 is TRUE (true mean = 0.4) ──────────────────────
type2 <- 0
for (i in 1:TRIALS) {
  s <- rnorm(n, 0.4, 1)                  # H1 true: mean = 0.4
  z <- mean(s) / (sd(s) / sqrt(n))
  if (abs(z) <= 1.96) type2 <- type2 + 1 # fail to reject
}
beta <- type2 / TRIALS
cat(sprintf("Type II error rate: %.4f\n", beta))
cat(sprintf("Power (1 - beta)  : %.4f   (probability of detecting the real effect)\n", 1 - beta))
cat("\nThe Type I rate matches alpha by construction. Power depends on effect\n")
cat("size, sample size, and alpha -- quantified directly in Part 3.\n")

**Common mistakes:**
- Reporting "p = 0.03 means 97% chance the effect is real" -- false; the p-value says nothing directly about P(H1)
- Treating "fail to reject" as "proved H0 true" -- absence of evidence is not evidence of absence; it may just be low power
- Forgetting that a high Type I rate is the *defined* behavior at the chosen alpha -- 5% of true nulls will be rejected by design

---
## SS4 -- The Peeking Problem

**Intuition:** In an A/B test it is tempting to check results continuously and stop
as soon as you see significance. This **inflates the false-positive rate
dramatically**. Each peek is another chance to cross the significance threshold by
luck. If you peek 20 times, your real Type I error rate can be 4-5x the nominal 5%.
This is one of the most consequential and frequently-tested ideas in industry
experimentation.

**Why it happens:** the significance threshold assumes a *single* test. Repeated
looks at accumulating data are multiple correlated tests -- the math no longer holds.

---

> **Interview prompt:** *"An analyst checks their A/B test every day and stops when
> p < 0.05. What's wrong with this? Demonstrate it by simulation."*

In [ ]:
# ── Simulate experiments where H0 is ALWAYS TRUE (no real effect) ────────────
experiment_rejects <- function(peek, seed) {
  # Run an experiment of 200 observations under H0 (no effect).
  # peek=TRUE  : test repeatedly as data accumulates, stop if ever significant.
  # peek=FALSE : test ONCE at the end (the correct procedure).
  # Returns TRUE if H0 was (wrongly) rejected.
  set.seed(seed)
  data <- rnorm(200, 0.0, 1)             # H0 TRUE: mean = 0
  if (peek) {
    for (n_so_far in seq(20, 200, by = 10)) {   # peek every 10 obs
      s <- data[1:n_so_far]
      z <- mean(s) / (sd(s) / sqrt(n_so_far))
      if (abs(z) > 1.96) return(TRUE)            # stop early, declare winner
    }
    return(FALSE)
  } else {
    z <- mean(data) / (sd(data) / sqrt(200))
    return(abs(z) > 1.96)
  }
}

REPS <- 3000
fp_peek   <- mean(sapply(1:REPS, function(i) experiment_rejects(TRUE,  i)))
fp_single <- mean(sapply(1:REPS, function(i) experiment_rejects(FALSE, i)))

cat("False-positive rate (H0 is true in every experiment):\n")
cat(sprintf("  WITH peeking (test every 10 obs) : %.1f%%   <- massively inflated\n", fp_peek * 100))
cat(sprintf("  WITHOUT peeking (one test at end): %.1f%%   <- correct, ~5%%\n", fp_single * 100))
cat(sprintf("\n  Peeking inflated the error rate by ~%.1fx.\n", fp_peek / fp_single))
cat("  Fixes: fix the sample size in advance, or use sequential-testing\n")
cat("  corrections (alpha-spending, always-valid p-values).\n")

**Common mistakes:**
- Stopping an experiment the moment it hits significance -- this is peeking, and it manufactures false positives
- Restarting or extending an experiment because the result "wasn't quite significant" -- same problem in reverse
- Not fixing the sample size (or a valid sequential procedure) before launching

**The fix:** decide the sample size in advance (Part 3), or use methods designed for continuous monitoring (sequential testing, Bayesian approaches).

---
## SS5 -- One-Sample Tests from Scratch

**Intuition:** Test whether a sample's mean differs from a hypothesized value
$\mu_0$. Use a **z-test** if the population variance is known (rare) or n is large;
use a **t-test** when you estimate the variance from the sample (the usual case).
The test statistic measures how many standard errors the sample mean sits from $\mu_0$.

$$z = \frac{\bar{x} - \mu_0}{\sigma/\sqrt{n}}, \qquad t = \frac{\bar{x} - \mu_0}{s/\sqrt{n}}$$

---

> **Interview prompt:** *"Implement a one-sample t-test from scratch (statistic and
> p-value) using base R, and check it against `t.test()`."*

In [ ]:
set.seed(42)

# Sample with true mean 52; test against H0: mu = 50
data <- rnorm(60, 52, 10)
mu_0 <- 50
n    <- length(data)

# ── One-sample z-test (treats sigma as known / large n) ──────────────────────
one_sample_z <- function(data, mu0, sigma) {
  z <- (mean(data) - mu0) / (sigma / sqrt(length(data)))
  list(stat = z, p = norm_two_sided_p(z))
}

# ── One-sample t-test (variance estimated from the sample) ───────────────────
one_sample_t <- function(data, mu0) {
  n  <- length(data)
  se <- sd(data) / sqrt(n)
  t  <- (mean(data) - mu0) / se
  list(stat = t, p = t_two_sided_p(t, df = n - 1))
}

z_res <- one_sample_z(data, mu_0, sigma = 10)
t_res <- one_sample_t(data, mu_0)

cat(sprintf("Sample mean: %.3f  (H0: mu = %s)\n", mean(data), mu_0))
cat(sprintf("\n%-16s %10s %10s %12s\n", "Test", "Statistic", "p-value", "Reject H0?"))
cat(sprintf("%-16s %10.4f %10.4f %12s\n", "z-test", z_res$stat, z_res$p, z_res$p < 0.05))
cat(sprintf("%-16s %10.4f %10.4f %12s\n", "t-test", t_res$stat, t_res$p, t_res$p < 0.05))
cat("\nThe t-test is the honest choice here -- we estimated the SD from the data.\n")

# Sanity check against base R's built-in
cat("\nVerify against t.test():\n")
print(t.test(data, mu = mu_0))

**Common mistakes:**
- Using a z-test when the variance is estimated from the sample -- technically wrong, though the gap shrinks as n grows
- Reporting a one-sided p-value without pre-registering the direction -- default to two-sided
- Confusing the standard deviation (`sd(data)`) with the standard error (`sd(data)/sqrt(n)`) in the denominator
- Hand-rolling this in production R code -- `t.test()` is built in and battle-tested; reach for it directly once you understand what it's doing

---
## SS6 -- Two-Sample Tests and Effect Size

**Intuition:** Compare two groups. For means, **Welch's t-test** is the industry
default -- unlike Student's t-test, it does *not* assume the two groups have equal
variance, so it is safer with no real downside. For conversion rates (binary
outcomes), use the **two-proportion z-test**.

But a p-value alone is incomplete. Always pair it with an **effect size** --
how *big* is the difference? **Cohen's d** expresses the gap in standard-deviation
units; **relative lift** expresses it as a percentage change.

$$t_{Welch} = \frac{\bar{x}_b - \bar{x}_a}{\sqrt{s_a^2/n_a + s_b^2/n_b}}, \qquad d = \frac{\bar{x}_b - \bar{x}_a}{s_{pooled}}$$

This connects directly to SS9 of the sampling notebook, which solved the same problem
with the bootstrap and permutation tests. Here are the parametric versions.

---

> **Interview prompt:** *"Implement Welch's t-test and a two-proportion z-test from
> scratch. Report effect sizes alongside the p-values. Compare to `t.test()` and
> `prop.test()`."*

In [ ]:
set.seed(42)

# ── Continuous outcome: Welch's t-test + Cohen's d ───────────────────────────
group_a <- rnorm(200, 50, 10)
group_b <- rnorm(180, 53, 12)             # different mean AND variance

welch_t_test <- function(a, b) {
  # Welch's two-sample t-test (unequal variances). Returns list(t, df, p).
  na <- length(a); nb <- length(b)
  va <- var(a); vb <- var(b)
  se <- sqrt(va / na + vb / nb)
  t  <- (mean(b) - mean(a)) / se
  # Welch-Satterthwaite degrees of freedom
  df <- (va / na + vb / nb)^2 / ((va / na)^2 / (na - 1) + (vb / nb)^2 / (nb - 1))
  list(t = t, df = df, p = t_two_sided_p(t, df))
}

cohens_d <- function(a, b) {
  na <- length(a); nb <- length(b)
  pooled_sd <- sqrt(((na - 1) * var(a) + (nb - 1) * var(b)) / (na + nb - 2))
  (mean(b) - mean(a)) / pooled_sd
}

res <- welch_t_test(group_a, group_b)
d   <- cohens_d(group_a, group_b)
cat("Continuous outcome -- Welch's t-test:\n")
cat(sprintf("  mean A = %.2f, mean B = %.2f\n", mean(group_a), mean(group_b)))
cat(sprintf("  t = %.4f, df = %.1f, p = %.4f\n", res$t, res$df, res$p))
cat(sprintf("  Cohen's d = %.3f  (%s effect)\n", d,
            if (abs(d) < 0.5) "small" else if (abs(d) < 0.8) "medium" else "large"))
cat("\n  Verify against base R's t.test(var.equal = FALSE) -- this IS Welch's test:\n")
print(t.test(group_b, group_a, var.equal = FALSE))

# ── Binary outcome: two-proportion z-test + relative lift ────────────────────
conv_a <- rbinom(500, 1, 0.08)            # control: 8%
conv_b <- rbinom(500, 1, 0.11)            # treatment: 11%

two_proportion_z <- function(a, b) {
  # Two-proportion z-test (pooled). Returns list(z, p).
  na <- length(a); nb <- length(b)
  pa <- mean(a); pb <- mean(b)
  p_pool <- (sum(a) + sum(b)) / (na + nb)
  se <- sqrt(p_pool * (1 - p_pool) * (1/na + 1/nb))
  z  <- (pb - pa) / se
  list(z = z, p = norm_two_sided_p(z))
}

prop_res <- two_proportion_z(conv_a, conv_b)
lift <- (mean(conv_b) - mean(conv_a)) / mean(conv_a)
cat("\nBinary outcome -- two-proportion z-test:\n")
cat(sprintf("  rate A = %.4f, rate B = %.4f\n", mean(conv_a), mean(conv_b)))
cat(sprintf("  z = %.4f, p = %.4f\n", prop_res$z, prop_res$p))
cat(sprintf("  Relative lift = %+.1f%%  <- the number the business actually cares about\n", lift * 100))
cat("\n  Verify against base R's prop.test() (chi-squared form of the same test):\n")
print(prop.test(c(sum(conv_a), sum(conv_b)), c(length(conv_a), length(conv_b))))

**Effect size reference (Cohen's d):** ~0.2 small, ~0.5 medium, ~0.8 large.

**Common mistakes:**
- Defaulting to Student's t-test (equal-variance) -- Welch's is safer and nearly always preferred in practice; in R, remember `var.equal = FALSE` is `t.test()`'s **default**, so plain `t.test(a, b)` already gives you Welch's test
- Reporting statistical significance without effect size -- a tiny, useless difference can be "significant" at large n
- Using a t-test on binary conversion data -- use the two-proportion z-test (`prop.test()` in R)
- Confusing absolute difference (3 percentage points) with relative lift (+37.5%) -- be explicit about which

---
## SS7 -- Multiple Testing Correction

**Intuition:** Run 20 independent tests at $\alpha = 0.05$ and you expect one false
positive *even if nothing is real*. Testing many hypotheses inflates the chance of
spurious "discoveries." Two corrections:

- **Bonferroni:** compare each p-value to $\alpha/m$. Controls the probability of
  *any* false positive (Family-Wise Error Rate). Simple but conservative -- it
  misses real effects when m is large.
- **Benjamini-Hochberg (BH):** controls the *expected proportion* of false positives
  among rejections (False Discovery Rate). More powerful -- recovers more true
  effects while keeping false discoveries bounded. The industry default for
  large-scale testing.

R has `p.adjust()` built in for both methods -- the manual version below shows what
it does internally.

---

> **Interview prompt:** *"You ran 20 A/B tests. Implement Bonferroni and
> Benjamini-Hochberg correction from scratch, then verify against `p.adjust()`."*

In [ ]:
# ── Simulate 20 tests: 8 have a real effect, 12 are null ─────────────────────
set.seed(0)
m <- 20
n_real <- 8
p_values <- numeric(m); is_real <- logical(m)
for (i in 1:m) {
  if (i <= n_real) {
    s <- rnorm(80, 0.45, 1)               # real effect
    is_real[i] <- TRUE
  } else {
    s <- rnorm(80, 0.0, 1)                # null
    is_real[i] <- FALSE
  }
  z <- mean(s) / (sd(s) / sqrt(80))
  p_values[i] <- norm_two_sided_p(z)
}

# ── No correction ─────────────────────────────────────────────────────────────
raw_reject <- p_values < 0.05

# ── Bonferroni: threshold alpha / m ──────────────────────────────────────────
bonf_reject <- p_values < (0.05 / m)

# ── Benjamini-Hochberg: step-up procedure (manual, for understanding) ────────
benjamini_hochberg <- function(p_values, alpha = 0.05) {
  m      <- length(p_values)
  order_ <- order(p_values)                          # ascending
  ranked <- p_values[order_]
  thresh <- (seq_len(m) / m) * alpha                  # i/m * alpha
  below  <- ranked <= thresh
  reject <- rep(FALSE, m)
  if (any(below)) {
    k <- max(which(below))                            # largest passing rank
    reject[order_[1:k]] <- TRUE                        # reject all up to k
  }
  reject
}

bh_reject <- benjamini_hochberg(p_values, alpha = 0.05)

# ── Compare ───────────────────────────────────────────────────────────────────
summarize <- function(name, rej) {
  tp <- sum(rej & is_real)
  fp <- sum(rej & !is_real)
  cat(sprintf("  %-22s rejected %2d  ->  %d true, %d false\n", name, sum(rej), tp, fp))
}

cat(sprintf("Ground truth: %d real effects out of %d tests\n\n", n_real, m))
summarize("No correction", raw_reject)
summarize("Bonferroni (FWER)", bonf_reject)
summarize("Benjamini-Hochberg (FDR)", bh_reject)
cat("\nBonferroni is conservative -- it misses real effects to avoid any false\n")
cat("positive. BH recovers more true effects while controlling the false-discovery\n")
cat("proportion. BH is the default when you run many tests.\n")

# Verify against base R's built-in
cat("\nVerify against p.adjust():\n")
cat("  Bonferroni match:", all((p.adjust(p_values, method = "bonferroni") < 0.05) == bonf_reject), "\n")
cat("  BH match        :", all((p.adjust(p_values, method = "BH") < 0.05) == bh_reject), "\n")

**Bonferroni vs. BH:**

| | Controls | Power | Use when |
| :--- | :--- | :--- | :--- |
| Bonferroni | Family-Wise Error Rate (any false positive) | Lower | Few tests, false positives very costly |
| Benjamini-Hochberg | False Discovery Rate (proportion of false positives) | Higher | Many tests, some false positives tolerable |

**Common mistakes:**
- Running many tests and reporting raw p < 0.05 -- guarantees false discoveries
- Applying Bonferroni to hundreds of tests -- so conservative you miss almost everything; use BH
- Implementing BH as a simple threshold -- it is a *step-up* procedure: find the largest rank that passes, then reject everything below it
- Hand-rolling either correction in production code -- `p.adjust(p_values, method = "bonferroni"|"BH")` is base R and already correct

---
# Part 3 -- Experimental Design

---
## SS8 -- Statistical Power via Simulation

**Intuition:** Power is the probability your test detects a real effect of a given
size. Rather than memorize a formula (which needs distributional assumptions), you
can **simulate** it directly: generate data under the alternative many times, run
the test each time, and count how often it correctly rejects. This is intuitive,
pure base R, and impressive in an interview.

Power increases with: larger effect size, larger sample size, larger alpha.

---

> **Interview prompt:** *"Estimate the power of a two-sample test to detect an effect
> of 0.4 standard deviations at various sample sizes, using simulation."*

In [ ]:
simulate_power <- function(effect_size, n_per_group, alpha = 0.05, B = 2000, seed = 0) {
  # Estimate power by simulation: fraction of experiments that detect the effect.
  # effect_size : true difference in means (in SD units, since SD=1 here)
  set.seed(seed)
  z_crit <- if (alpha == 0.05) 1.96 else NA   # 95% two-sided
  rejections <- 0
  for (i in 1:B) {
    a <- rnorm(n_per_group, 0.0,         1)   # control
    b <- rnorm(n_per_group, effect_size, 1)   # treatment
    se <- sqrt(var(a) / n_per_group + var(b) / n_per_group)
    z  <- (mean(b) - mean(a)) / se
    if (abs(z) > z_crit) rejections <- rejections + 1
  }
  rejections / B
}

cat("Power to detect an effect of 0.4 SD (two-sample, alpha=0.05):\n")
cat(sprintf("  %12s %8s\n", "n per group", "power"))
for (n in c(20, 50, 100, 150, 200)) {
  pw  <- simulate_power(0.4, n)
  bar <- strrep("#", as.integer(pw * 40))
  cat(sprintf("  %12d %8.3f  %s\n", n, pw, bar))
}
cat("\nPower climbs with sample size. The conventional target is 0.80 --\n")
cat("an 80%% chance of detecting the effect if it is really there.\n")

**Common mistakes:**
- Running an underpowered experiment -- if power is 0.4, you'll miss a real effect more often than you catch it, and a null result tells you almost nothing
- Confusing power with significance -- alpha controls false positives (Type I); power controls false negatives (Type II)
- Forgetting that observed/post-hoc power computed from your actual result is circular and uninformative -- power is a *design-time* quantity

---
## SS9 -- Sample Size and Minimum Detectable Effect

**Intuition:** Two sides of the same coin, both answerable by inverting the power
simulation from SS8:

- **Sample size:** "I want 80% power to detect an effect of 0.4 -- how many users
  do I need?" Search upward in n until simulated power hits the target.
- **Minimum Detectable Effect (MDE):** "I can only get 100 users per group -- what's
  the smallest effect I can reliably detect?" Search across effect sizes at fixed n.

These are the questions product teams ask *before* launching an experiment.

---

> **Interview prompt:** *"How many samples per group do you need for 80% power to
> detect a 0.4 SD effect? And given 100 per group, what's your MDE?"*

In [ ]:
# (reuses simulate_power from the previous cell)

# ── Required sample size for a target power ───────────────────────────────────
required_n <- function(effect_size, target_power = 0.80, alpha = 0.05, seed = 0) {
  n <- 10
  while (n <= 5000) {
    if (simulate_power(effect_size, n, alpha, seed = seed) >= target_power) return(n)
    n <- n + 10
  }
  NA
}

# ── Minimum detectable effect at a fixed sample size ─────────────────────────
minimum_detectable_effect <- function(n_per_group, target_power = 0.80, alpha = 0.05, seed = 0) {
  es <- 0.05
  while (es <= 2.0) {
    if (simulate_power(es, n_per_group, alpha, seed = seed) >= target_power) return(round(es, 2))
    es <- es + 0.05
  }
  NA
}

n_needed <- required_n(0.4, target_power = 0.80)
mde_100  <- minimum_detectable_effect(100, target_power = 0.80)

cat("Experiment planning (two-sample, alpha=0.05, target power=0.80):\n")
cat(sprintf("  To detect effect 0.4  -> need n = %s per group\n", n_needed))
cat(sprintf("  Given n = 100/group   -> MDE = %s SD\n", mde_100))
cat("\nConsistency check: effect 0.4 needs ~100/group, and 100/group detects\n")
cat("~0.4 -- the two calculations agree, as they should.\n")

**Common mistakes:**
- Launching without a sample-size calculation, then peeking until significant (see SS4) -- the two mistakes compound
- Powering for a tiny effect that isn't practically meaningful -- requires enormous samples for no business value; power for the *minimum effect worth acting on*
- Ignoring that real A/B tests need the sample size *per variant*, and that multiple variants or metrics require further correction (SS7)

---
## SS10 -- End-to-End A/B Test Design

**Intuition:** Putting Parts 1-3 together into the workflow a data scientist actually
runs: decide the sample size *before* launching, collect exactly that much data,
run *one* test at the end, and report significance **and** effect size with a
confidence interval. This is the disciplined alternative to "launch, peek, stop when
green" -- which SS4 showed manufactures false positives.

---

> **Interview prompt:** *"Design and analyze a complete A/B test for a checkout
> conversion change. Current rate is 8%; you want 80% power to detect a 2 percentage
> point lift. Walk through sample size, then analyze the result."*

In [ ]:
set.seed(42)

# ── STEP 1: design -- required sample size BEFORE launch ─────────────────────
baseline_rate <- 0.08
target_lift   <- 0.02                                  # detect 8% -> 10%
treatment_rate_assumed <- baseline_rate + target_lift

required_n_proportions <- function(p0, p1, target_power = 0.80, alpha = 0.05, B = 1500, seed = 0) {
  n <- 100
  while (n <= 50000) {
    set.seed(seed)
    rej <- 0
    for (i in 1:B) {
      a <- rbinom(n, 1, p0)
      b <- rbinom(n, 1, p1)
      p_pool <- (sum(a) + sum(b)) / (2 * n)
      se <- sqrt(p_pool * (1 - p_pool) * (2 / n))
      if (se > 0 && abs((mean(b) - mean(a)) / se) > 1.96) rej <- rej + 1
    }
    if (rej / B >= target_power) return(n)
    n <- n + 500
  }
  NA
}

n_design <- required_n_proportions(baseline_rate, treatment_rate_assumed)
cat("STEP 1 -- Design\n")
cat(sprintf("  Baseline: %.0f%%, target lift: +%.0f%% (detect %.0f%%)\n",
            baseline_rate * 100, target_lift * 100, treatment_rate_assumed * 100))
cat(sprintf("  Required sample size: %s per variant for 80%% power\n", format(n_design, big.mark = ",")))

# ── STEP 2: collect exactly n_design per variant (simulate the real test) ────
# Suppose the TRUE treatment effect is a ~2.8pt lift (above our 2pt MDE)
true_treatment_rate <- 0.108
set.seed(43)   # seed for the data-collection draw
control   <- rbinom(n_design, 1, baseline_rate)
treatment <- rbinom(n_design, 1, true_treatment_rate)

# ── STEP 3: analyze ONCE at the end ──────────────────────────────────────────
two_proportion_z <- function(a, b) {
  na <- length(a); nb <- length(b)
  p_pool <- (sum(a) + sum(b)) / (na + nb)
  se <- sqrt(p_pool * (1 - p_pool) * (1/na + 1/nb))
  (mean(b) - mean(a)) / se
}

z    <- two_proportion_z(control, treatment)
p    <- norm_two_sided_p(z)
diff <- mean(treatment) - mean(control)
lift <- diff / mean(control)

# CI on the difference (bootstrap, from the sampling notebook's toolkit)
boot <- replicate(5000,
  mean(sample(treatment, length(treatment), replace = TRUE)) -
  mean(sample(control,   length(control),   replace = TRUE))
)
ci <- quantile(boot, c(0.025, 0.975))

cat("\nSTEP 2-3 -- Collect & Analyze (one test at the end)\n")
cat(sprintf("  Control   conversion: %.4f\n", mean(control)))
cat(sprintf("  Treatment conversion: %.4f\n", mean(treatment)))
cat(sprintf("  Absolute difference : %+.4f  (%+.2f pp)\n", diff, diff * 100))
cat(sprintf("  Relative lift       : %+.1f%%\n", lift * 100))
cat(sprintf("  z = %.3f,  p = %.4f  ->  %s\n", z, p, if (p < 0.05) "SIGNIFICANT" else "not significant"))
cat(sprintf("  95%% CI on difference: (%+.4f, %+.4f)\n", ci[1], ci[2]))
cat("\nDecision: report significance, effect size, AND the CI. The CI shows the\n")
cat("plausible range of the true lift -- essential for the business decision,\n")
cat("not just a yes/no on significance.\n")

**The disciplined A/B workflow:**
1. **Design first** -- compute required sample size for your minimum meaningful effect (SS9)
2. **Collect** exactly that much data -- no early stopping (SS4)
3. **Test once** at the end with the right test (SS6)
4. **Report** significance *and* effect size *and* a confidence interval (SS2, SS6)
5. **Correct** for multiplicity if running several variants or metrics (SS7)

**Why this matters:** every shortcut -- peeking, skipping the power calculation,
reporting p-values without effect sizes -- systematically corrupts the conclusion.
The discipline *is* the method.

---

## Connection to the Sampling Notebook

| This notebook (inference) | Sampling notebook |
| :--- | :--- |
| SS2 CI -- unified z/t/bootstrap | Bootstrap CI scattered across methods |
| SS6 Two-sample parametric tests | SS9 Two-sample bootstrap & permutation |
| SS7 Multiple testing correction | SS9 Single A/B test |
| SS8-9 Power & sample size | SS2 Stratified split (who's in the test) |
| SS10 A/B design capstone | Pulls in `sample()`, stratification, bootstrap |

Together they cover the full loop: **design the sample -> collect it -> infer from it.**